# Chapter 51 — Batching for Language Models

## Learning goals

Chapter 50 trained an MLP language model using every available example at each step.

This chapter replaces that full-batch approach with randomly sampled mini-batches.

By the end of this chapter, you should be able to:

- Define batch size, context length, and a valid starting position.
- Build aligned input and target batches directly from token IDs.
- Write and validate a reproducible `get_batch` function.
- Decode sampled rows to check their next-token targets.
- Explain why random starting positions preserve sequence order while shuffled text destroys it.
- Use random mini-batches in an MLP language model training loop.

## The big idea

One character-level training example pairs a fixed context with the character immediately after it.

For example, `"the " → "c"` uses four input characters and one target character.

A batch groups several such examples into rectangular tensors:

```text
input IDs:  [batch_size, context_length]
target IDs: [batch_size]
```

The row index preserves the pairing between each context and its target.

## Terms used in this chapter

- A **batch** is a group of examples processed together.
- **Batch size** is the number of examples, or rows, in one batch.
- **Context length** is the number of consecutive input tokens in each row.
- A **starting position** is the sequence index where a context begins.
- A **valid starting position** leaves room for the complete context and its following target.
- A **mini-batch** contains fewer examples than the complete training set.
- **Sampling with replacement** allows the same starting position to be selected more than once.

## Encode a small character corpus

Import PyTorch, stay on the CPU, and encode a short sentence whose windows are easy to inspect.

Sorting the vocabulary makes the character-to-ID mapping deterministic.

In [1]:
import torch

device = "cpu"
training_text = "the cat sat on the mat."
vocabulary = sorted(set(training_text))
character_to_id = {character: token_id for token_id, character in enumerate(vocabulary)}
id_to_character = {
    token_id: character for character, token_id in character_to_id.items()
}
token_ids = torch.tensor(
    [character_to_id[character] for character in training_text],
    dtype=torch.long,
    device=device,
)
decoded_text = "".join(id_to_character[int(token_id)] for token_id in token_ids)

print("device:", device)
print("training text:", repr(training_text))
print("number of tokens:", token_ids.shape[0])
print("vocabulary:", vocabulary)
print("token IDs:", token_ids)
print("decoded again:", decoded_text)

device: cpu
training text: 'the cat sat on the mat.'
number of tokens: 23
vocabulary: [' ', '.', 'a', 'c', 'e', 'h', 'm', 'n', 'o', 's', 't']
token IDs: tensor([10,  5,  4,  0,  3,  2, 10,  0,  9,  2, 10,  0,  8,  7,  0, 10,  5,  4,
         0,  6,  2, 10,  1])
decoded again: the cat sat on the mat.


The decoded text exactly matches the source, so the encoding retained spaces, punctuation, and sequence order.

## Build one example from a starting position

With context length four, starting position zero selects IDs for `"the "` and the following ID for `"c"`.

In [2]:
context_length = 4
start_index = 0
stop_index = start_index + context_length

input_ids = token_ids[start_index:stop_index]
target_id = token_ids[stop_index]

decoded_input = "".join(id_to_character[int(token_id)] for token_id in input_ids)
decoded_target = id_to_character[int(target_id)]

print("input IDs:", input_ids)
print("target ID:", target_id)
print("decoded example:", repr(decoded_input), "→", repr(decoded_target))

input IDs: tensor([10,  5,  4,  0])
target ID: tensor(3)
decoded example: 'the ' → 'c'


The target sits at `start_index + context_length`, immediately outside the input slice.

## Stack hand-picked examples

Start positions `0`, `4`, `8`, and `12` produce the four examples from the chapter introduction.

A helper keeps the slicing rule in one place and rejects indexes that cannot supply a complete example.

In [3]:
def build_batch_from_start_indexes(
    token_ids: torch.Tensor,
    start_indexes: torch.Tensor,
    context_length: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    if token_ids.ndim != 1:
        raise ValueError("token_ids must be a one-dimensional tensor.")
    if start_indexes.ndim != 1:
        raise ValueError("start_indexes must be a one-dimensional tensor.")
    if start_indexes.numel() == 0:
        raise ValueError("start_indexes must contain at least one index.")
    if context_length < 1:
        raise ValueError("context_length must be at least 1.")

    number_of_valid_start_positions = token_ids.shape[0] - context_length
    if number_of_valid_start_positions < 1:
        raise ValueError("token_ids must contain more tokens than context_length.")
    if torch.any(start_indexes < 0) or torch.any(
        start_indexes >= number_of_valid_start_positions
    ):
        raise ValueError("Every start index must identify a complete example.")

    input_rows: list[torch.Tensor] = []
    target_rows: list[torch.Tensor] = []

    for start_index in start_indexes.tolist():
        stop_index = start_index + context_length
        input_rows.append(token_ids[start_index:stop_index])
        target_rows.append(token_ids[stop_index])

    return torch.stack(input_rows), torch.stack(target_rows)


selected_start_indexes = torch.tensor([0, 4, 8, 12], device=device)
input_batch, target_batch = build_batch_from_start_indexes(
    token_ids=token_ids,
    start_indexes=selected_start_indexes,
    context_length=context_length,
)

print("input batch IDs:")
print(input_batch)
print("target batch IDs:")
print(target_batch)
print("input shape:", input_batch.shape)
print("target shape:", target_batch.shape)

input batch IDs:
tensor([[10,  5,  4,  0],
        [ 3,  2, 10,  0],
        [ 9,  2, 10,  0],
        [ 8,  7,  0, 10]])
target batch IDs:
tensor([3, 9, 8, 5])
input shape: torch.Size([4, 4])
target shape: torch.Size([4])


The input shape `[4, 4]` means four examples with four context IDs each.

The target shape `[4]` means one next-character ID for each input row.

## Decode the batch

Readable rows reveal alignment errors that correct tensor shapes cannot detect.

In [4]:
def decode_character_ids(
    token_ids: list[int],
    id_to_character: dict[int, str],
) -> str:
    return "".join(id_to_character[token_id] for token_id in token_ids)


def print_decoded_batch(
    input_batch: torch.Tensor,
    target_batch: torch.Tensor,
    id_to_character: dict[int, str],
) -> None:
    if input_batch.ndim != 2 or target_batch.ndim != 1:
        raise ValueError("Expected rank-two inputs and rank-one targets.")
    if input_batch.shape[0] != target_batch.shape[0]:
        raise ValueError("Input and target batches must have the same row count.")

    print("row | context | target")
    print("-" * 24)
    for row_index in range(input_batch.shape[0]):
        decoded_input = decode_character_ids(
            input_batch[row_index].tolist(),
            id_to_character,
        )
        target_id = int(target_batch[row_index].item())
        print(f"{row_index:>3} | {decoded_input!r:>9} | {id_to_character[target_id]!r}")


print_decoded_batch(input_batch, target_batch, id_to_character)

row | context | target
------------------------
  0 |    'the ' | 'c'
  1 |    'cat ' | 's'
  2 |    'sat ' | 'o'
  3 |    'on t' | 'h'


Every target is the original character immediately after its row's context.

The row index is therefore part of the data relationship and must stay aligned across both tensors.

## Find the valid starting positions

A sequence of `N` tokens and a context length of `C` has `N - C` valid starts.

The last valid start is `N - C - 1` because one target token must remain after the context.

In [5]:
number_of_tokens = token_ids.shape[0]
number_of_valid_start_positions = number_of_tokens - context_length
largest_valid_start_index = number_of_valid_start_positions - 1

print("number of tokens:", number_of_tokens)
print("context length:", context_length)
print("number of valid starts:", number_of_valid_start_positions)
print("largest valid start:", largest_valid_start_index)

assert token_ids[largest_valid_start_index + context_length].ndim == 0

number of tokens: 23
context length: 4
number of valid starts: 19
largest valid start: 18


The assertion confirms that the last valid context still has a scalar target at its end.

## Sample a random mini-batch

`get_batch` chooses valid starts with `torch.randint`, then delegates the slicing to the tested helper.

The optional generator makes stored examples reproducible without resetting unrelated global randomness.

In [6]:
def get_batch(
    token_ids: torch.Tensor,
    batch_size: int,
    context_length: int,
    generator: torch.Generator | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    if token_ids.ndim != 1:
        raise ValueError("token_ids must be a one-dimensional tensor.")
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")
    if context_length < 1:
        raise ValueError("context_length must be at least 1.")

    number_of_valid_start_positions = token_ids.shape[0] - context_length
    if number_of_valid_start_positions < 1:
        raise ValueError("token_ids must contain more tokens than context_length.")

    start_indexes = torch.randint(
        low=0,
        high=number_of_valid_start_positions,
        size=(batch_size,),
        generator=generator,
        device=token_ids.device,
    )
    return build_batch_from_start_indexes(
        token_ids=token_ids,
        start_indexes=start_indexes,
        context_length=context_length,
    )


batch_generator = torch.Generator(device=device).manual_seed(51)
input_batch, target_batch = get_batch(
    token_ids=token_ids,
    batch_size=4,
    context_length=context_length,
    generator=batch_generator,
)

print("input batch IDs:")
print(input_batch)
print("target batch IDs:")
print(target_batch)
print("input shape:", input_batch.shape)
print("target shape:", target_batch.shape)

input batch IDs:
tensor([[ 8,  7,  0, 10],
        [ 0, 10,  5,  4],
        [ 4,  0,  6,  2],
        [ 2, 10,  0,  8]])
target batch IDs:
tensor([ 5,  0, 10,  7])
input shape: torch.Size([4, 4])
target shape: torch.Size([4])


Decode the sampled tensors themselves rather than reconstructing examples from a separate list.

In [7]:
print_decoded_batch(input_batch, target_batch, id_to_character)

row | context | target
------------------------
  0 |    'on t' | 'h'
  1 |    ' the' | ' '
  2 |    'e ma' | 't'
  3 |    'at o' | 'n'


The selected windows vary, but each row still preserves the corpus order and uses its true next character.

Because `torch.randint` samples with replacement, a batch may contain a repeated row.

## Random starts preserve sequence order

Randomness should choose which intact windows appear in a mini-batch.

It should not reorder the characters inside the corpus or inside a window.

The following contrast intentionally creates corrupted text only to show why character shuffling is unsuitable training data.

In [8]:
shuffle_generator = torch.Generator(device=device).manual_seed(51)
random_permutation = torch.randperm(
    len(training_text),
    generator=shuffle_generator,
    device=device,
)
shuffled_text = "".join(training_text[index] for index in random_permutation.tolist())

print("original text:", repr(training_text))
print("shuffled text:", repr(shuffled_text))

original text: 'the cat sat on the mat.'
shuffled text: '.shmtaaoet  c n hettt a'


The shuffled string contains the same character counts but no longer preserves the original next-character relationships.

A language model trained on those shuffled transitions would learn the corrupted order rather than the source sequence.

## Reject invalid requests

Clear errors at the batch boundary are easier to diagnose than empty slices or failed `torch.stack` calls later.

In [9]:
invalid_requests = [
    {"batch_size": 0, "context_length": 4},
    {"batch_size": 2, "context_length": 0},
    {"batch_size": 2, "context_length": len(token_ids)},
]

for request in invalid_requests:
    try:
        get_batch(token_ids=token_ids, **request)
    except ValueError as error:
        print(f"{request}: {error}")

{'batch_size': 0, 'context_length': 4}: batch_size must be at least 1.
{'batch_size': 2, 'context_length': 0}: context_length must be at least 1.
{'batch_size': 2, 'context_length': 23}: token_ids must contain more tokens than context_length.


The checks cover non-positive dimensions and sequences that cannot provide both a full context and a target.

## Send a batch through the Chapter 50 MLP

Reuse the same fixed-context architecture to connect the new data loader with model input and output shapes.

The model returns one row of vocabulary logits for every sampled context.

In [10]:
from typing import cast


class MlpLanguageModel(torch.nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        context_length: int,
        embedding_dimension: int,
        hidden_size: int,
    ) -> None:
        super().__init__()

        self.context_length = context_length
        self.embedding_dimension = embedding_dimension
        self.token_embedding_table = torch.nn.Embedding(
            vocabulary_size,
            embedding_dimension,
        )
        self.first_linear_layer = torch.nn.Linear(
            context_length * embedding_dimension,
            hidden_size,
        )
        self.nonlinearity = torch.nn.Tanh()
        self.output_layer = torch.nn.Linear(hidden_size, vocabulary_size)

    def forward(self, input_token_ids: torch.Tensor) -> torch.Tensor:
        token_embeddings = self.token_embedding_table(input_token_ids)
        flattened_embeddings = token_embeddings.reshape(
            input_token_ids.shape[0],
            self.context_length * self.embedding_dimension,
        )
        hidden_values = self.first_linear_layer(flattened_embeddings)
        activated_hidden_values = self.nonlinearity(hidden_values)
        return cast(torch.Tensor, self.output_layer(activated_hidden_values))


torch.manual_seed(51)
model = MlpLanguageModel(
    vocabulary_size=len(vocabulary),
    context_length=context_length,
    embedding_dimension=8,
    hidden_size=32,
).to(device)
next_token_logits = model(input_batch)
loss_function = torch.nn.CrossEntropyLoss()
batch_loss = loss_function(next_token_logits, target_batch)

print("input shape:", input_batch.shape)
print("target shape:", target_batch.shape)
print("logit shape:", next_token_logits.shape)
print("batch loss:", batch_loss.item())

input shape: torch.Size([4, 4])
target shape: torch.Size([4])
logit shape: torch.Size([4, 11])
batch loss: 2.4822239875793457


The logits have shape `[batch_size, vocabulary_size]`, while cross-entropy reduces the four row losses to one scalar mean.

## Train with random mini-batches

Repeat a small patterned corpus to provide many valid starts, then sample a fresh mini-batch at every update.

This remains a toy fitting exercise rather than evidence of general language ability.

In [11]:
larger_training_text = (
    "the cat sat on the mat. "
    "the dog sat on the rug. "
    "the cat ran to the dog. "
    "the dog ran to the cat. "
) * 20
larger_vocabulary = sorted(set(larger_training_text))
larger_character_to_id = {
    character: token_id for token_id, character in enumerate(larger_vocabulary)
}
larger_token_ids = torch.tensor(
    [larger_character_to_id[character] for character in larger_training_text],
    dtype=torch.long,
    device=device,
)

torch.manual_seed(51)
model = MlpLanguageModel(
    vocabulary_size=len(larger_vocabulary),
    context_length=context_length,
    embedding_dimension=8,
    hidden_size=64,
).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.03,
    weight_decay=0.0,
)
training_generator = torch.Generator(device=device).manual_seed(51)
batch_size = 16
number_of_steps = 300
training_losses: list[float] = []

for _ in range(number_of_steps):
    input_batch, target_batch = get_batch(
        token_ids=larger_token_ids,
        batch_size=batch_size,
        context_length=context_length,
        generator=training_generator,
    )
    next_token_logits = model(input_batch)
    loss = loss_function(next_token_logits, target_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    training_losses.append(loss.item())

print("initial batch loss:", training_losses[0])
print("final batch loss:", training_losses[-1])
print("mean of first 20 losses:", sum(training_losses[:20]) / 20)
print("mean of final 20 losses:", sum(training_losses[-20:]) / 20)

initial batch loss: 2.774198055267334
final batch loss: 0.27777692675590515
mean of first 20 losses: 1.1417153418064117
mean of final 20 losses: 0.2251185913104564


Individual batch losses can move up or down because each step measures a different sample.

Comparing averages over several steps gives a more stable view, and the final 20-step mean should be lower for this deterministic run.

## Inspect a training batch

Decode one fresh batch from the larger corpus to verify that training still receives intact contexts and aligned targets.

In [12]:
larger_id_to_character = {
    token_id: character for character, token_id in larger_character_to_id.items()
}
inspection_generator = torch.Generator(device=device).manual_seed(52)
inspection_inputs, inspection_targets = get_batch(
    token_ids=larger_token_ids,
    batch_size=4,
    context_length=context_length,
    generator=inspection_generator,
)

print_decoded_batch(
    inspection_inputs,
    inspection_targets,
    larger_id_to_character,
)

row | context | target
------------------------
  0 |    'og r' | 'a'
  1 |    'to t' | 'h'
  2 |    'mat.' | ' '
  3 |    'g. t' | 'h'


The random rows come from different locations, but no row changes the character order found at its location.

## Batch size and context length have different costs

A tensor batch lets PyTorch apply the same model operations to many examples together instead of managing one example at a time.

A mini-batch update is also cheaper than recomputing the loss over a large complete dataset at every step.

Increasing **batch size** adds rows, so one update processes more examples and usually uses more memory.

Increasing **context length** adds input positions to every row and gives the model more previous tokens.

For the Chapter 50 MLP, a longer context also widens the first linear layer because its input size is `context_length × embedding_dimension`.

Small mini-batches use less memory but produce noisier estimates of the full-dataset loss.

Larger mini-batches produce more stable estimates but require more computation and memory per update.

## One target now, many targets next

This chapter's MLP predicts once after each complete context:

```text
inputs:  [batch_size, context_length]
targets: [batch_size]
```

GPT-style training predicts the next token at every context position:

```text
inputs:  [batch_size, context_length]
targets: [batch_size, context_length]
```

Both styles sample intact sequence windows, but they construct different target tensors and require different model output shapes.

## Common mistakes

- Do not confuse batch size, the number of rows, with context length, the number of input IDs per row.
- Do not select a start that lacks room for both the complete context and its target.
- Do not detach targets from their corresponding input row.
- Do not shuffle raw characters and call the result random batching.
- Do not assume repeated sampled rows indicate an error when sampling with replacement.
- Do not trust tensor shapes without decoding representative examples.
- Do not expect every random batch loss to be lower than the previous one.

## Takeaways

A one-target mini-batch contains context IDs shaped `[B, C]` and aligned next-token IDs shaped `[B]`.

For `N` tokens and context length `C`, valid starts range from `0` through `N - C - 1`.

`get_batch` can sample those starts with replacement and build a fresh mini-batch for each training step.

Randomly choosing windows is useful because every selected window preserves its original local order.

Randomly shuffling the underlying text is wrong because it destroys the next-token relationships the model should learn.

## What comes next

The next chapter returns to the GPT-style shifted targets introduced in Chapter 49 and makes them the main batch format.

That GPT-style batch layout prepares the data for transformer training while retaining the same rule that sampled windows must preserve sequence order.